Part 1 (Duplicates)  
First investigate the dataset to see if there are any duplicate houses. If there are duplicate houses, remove these from the dataset.

In [50]:
import pandas as pd
df = pd.read_csv("data/tacoma_houses.csv", low_memory=False)

df["house_number"] = df["address"].str.split().str[0]

Part 2 (Initial Data Cleaning Plan)
In this part of the project you will make a data cleaning plan:

1) Identify the variables from the dataset that you would like to keep and that you believe might be helpful when explaining a house's price and explain why you included this variable. You must identify at least 20 variables. You may identify more than this if you would like. The goal is to include as many features as you feel may be helpful.  
Inclusions: Reduces installations needed post purchase, thus increasing price/demand      
Total Bedrooms: Amount of rooms that can hold individuals, more rooms may indicate a higher price      
Garage Y/N: Type of garage may not play a large role but having either parking for vehicles or storage may be ideal        
Air Conditioning Y/N: Some people (i.e. me) cannot function in bolstering heat, PNW AC may not be a neccesity for some though      
Bathrooms: Indicates more space for more individuals      
Total Parking: Allows for more individuals to park, either in the houeshold or guests      
Site Features: More or higher-end features will be desirable for some, raising demand and thus price      
Interior Features: More features can directly raise the cost      
Security Features: Can help people feel safer    
house_size: Honestly this one I feel is self-explanatory as it is one of the key features listed on home purchasing sites      
Tax Annual Amount: A reduction in future prices may raise upfront price      
Year Built: Indicates age of house, and therefore potential failure points unless there has been a remodel or meticulous maintenance     
MLS Area: Certain areas may have a higher demand for several reasons (general safety, nearby features, view, etc.)      
Levels: Houses with more stories typically have more square footage, thus increasing price as stated earlier, and more stories can help reduce total area on the lot used for the house, thus increasing usable external property      
lot_size: Different sizes of lots are more desirable to different people (people who don't want to upkeep much = smaller lots are desiarable, people who loves outdoors work = larger lots are desirable, etc)    
Flooring: Can impact ease of cleaning and accessiblity    
HOA: Apologies for this but I have a personal vendetta against HOA's, and I know plenty others who feel the same, thus making a property not in an HOA more desirable      
HOA Fee: Same point with annual tax amount listed above    
Property Condition: Properties in worse conditions will require more money being used post sale to fix necesary issues but will lower upfront costs      
Foundation Details: Type of foundation indicates how well a house could stand over years of time and during natural disasters      
Lot Features: More features can increase the prices based on demand      

2) For each of these variables, look at how many missing values there are. If a variable does contain missing values, determine how you would like to handle these missing values and provide a justification for your approach. Approaches may include:  
Listwise deletion  
Variable deletion  
Case deletion  
Imputation  
Another method you may feel is appropriate  
Inclusions: If no input, inclusions are treated as having none      
Total Bedrooms: Add amount of bedrooms from upper and lower bedroom columns      
Garage Y/N: Refer to attached garage column, fill if value exists there, else no        
Air Conditioning Y/N: 1 value missing, okay to delete row        
Bathrooms: 1 value missing, okay to delete row      
Total Parking: Fill with median rounded down      
Site Features: If none are listed, okay to treat as having no site features      
Interior Features: Empty cells get filled with no interior features      
Security Features: Same with interior features    
house_size: No missing values      
Tax Annual Amount: Fill with mean      
Year Built: No missing values      
MLS Area: No missing values      
Levels: Pull from stories count, else fill with mode      
lot_size: No missing values    
Flooring: Fill with unknown    
HOA: Empty cells get filled with No      
HOA Fee: Fill empty cells with 0    
Property Condition: Fill with "Unknown"      
Foundation Details: Also fill with "Unknown"      
Lot Features: Empty cells get filled with No    



4) For each of the variables you decide to keep from your original list of 20 variables, determine what else needs to be done in order to prepare that data to be used in modeling. Some things to consider may include:  
Does text need to be removed?    
Does the data type need to be converted?    
Do units of measurement need to be standardized?      
Do categories for categorical variables need to be standardized?      
Any other special considerations?      
Inclusions: Remove plurals and standardize/map names      
Total Bedrooms: Already integers      
Garage Y/N: Convert no to 0 and yes to 1         
Air Conditioning Y/N: Convert no to 0 and yes to 1      
Bathrooms: Already floats      
Total Parking: Convert to integers      
Site Features: Remove commas and standardize/map names    
Interior Features: Same as Site Features      
Security Features: Map features into more general categories    
house_size: Remove commas, "SQFT" and convert to integers      
Tax Annual Amount: Remove dollar signs, commas, and whitespace, convert to float      
Year Built: Convert to a new "House Age" column by 2026 - Year Built      
MLS Area: Drop words, keep numeric code      
Levels: Standardize to floats      
lot_size: Convert to acres if needed, remove "Acres"    
Flooring: Standardize/map names    
HOA: Convert no to 0 and yes to 1      
HOA Fee: Remove dollar signs, whitespace, and convert to float    
Property Condition: Standarize and map to a numeric scale      
Foundation Details: Standardize/map names      
Lot Features: Same as Foundation Details    

Part 3 (Implement Data Cleaning Plan)
Implement your data cleaning plan and clean each of the variables that you decided to keep. By the time that you are done cleaning the dataset, you should make sure that every variable you selected could be easily represented as a number. This means that one of the following is true for each of the variables:

Any quantitative variables should either be an integer or a float
Any categorical variables should be cleaned so that a category doesn't have two or more names that are being use to refer to it (e.g., Blue and blue)
There shouldn't be any missing values for any of the variables

In [51]:
import numpy as np
import math
import re

#In complete transparancy, the following code is produced by AI in line with your regulations on AI usage on the project.
#I can verify what is happening, even for the sections that have not been covered in class with knowledge from prior experience and documemtation and I had Claude generate the code based off of my ideas and decisions
# ─────────────────────────────────────────────────────────────
# 1. INCLUSIONS
# ─────────────────────────────────────────────────────────────
# Fill missing with "None"
df['Inclusions'] = df['Inclusions'].fillna('None')

# Canonical name map — normalise plurals, spacing, and slash variants
inclusion_name_map = {
    'dishwasher(s)': 'Dishwasher', 'dishwashers': 'Dishwasher', 'dishwasher': 'Dishwasher',
    'dryer(s)': 'Dryer', 'dryers': 'Dryer', 'dryer': 'Dryer',
    'washer(s)': 'Washer', 'washers': 'Washer', 'washer': 'Washer',
    'microwave(s)': 'Microwave', 'microwaves': 'Microwave', 'microwave': 'Microwave',
    'refrigerator(s)': 'Refrigerator', 'refrigerators': 'Refrigerator', 'refrigerator': 'Refrigerator',
    "stove(s)/range(s)": 'StoveRange', 'stovesranges': 'StoveRange', 'stoverange': 'StoveRange',
    'range/oven': 'StoveRange', 'rangeoven': 'StoveRange',
    'garbage disposal': 'GarbageDisposal', 'garbagedisposal': 'GarbageDisposal', 'garbage': 'GarbageDisposal',
    'double oven': 'DoubleOven', 'doubleoven': 'DoubleOven',
    'trash compactor': 'TrashCompactor', 'trashcompactor': 'TrashCompactor',
    'leased equipment': 'LeasedEquipment', 'leasedequipment': 'LeasedEquipment',
    'security system': 'SecuritySystem', 'securitysystem': 'SecuritySystem',
    'above ground pool': 'AboveGroundPool', 'abovegroundpool': 'AboveGroundPool',
    'awnings': 'Awnings',
    'drapes': 'Drapes',
    'cable/tv': 'CableTV',
    'sewer': 'Sewer',
    'water': 'Water',
    'see remarks': None, 'seeremarks': None, 'seeremarks_': None,
    'none': None,
}

def normalise_inclusions(value):
    if pd.isna(value) or value.strip().lower() == 'none':
        return 'None'
    canonical_items = []
    for raw in value.split(','):
        key = raw.strip().lower()
        canonical = inclusion_name_map.get(key)
        if canonical:
            canonical_items.append(canonical)
    return ', '.join(sorted(set(canonical_items))) if canonical_items else 'None'

df['Inclusions'] = df['Inclusions'].apply(normalise_inclusions)


# ─────────────────────────────────────────────────────────────
# 2. TOTAL BEDROOMS  (re-derive from upper + lower)
# ─────────────────────────────────────────────────────────────
df['Bedrooms Upper'] = df['Bedrooms Upper'].fillna(0)
df['Bedrooms Lower'] = df['Bedrooms Lower'].fillna(0)
df['Total Bedrooms'] = (df['Bedrooms Upper'] + df['Bedrooms Lower']).astype(int)
df.drop(columns=['Bedrooms Upper', 'Bedrooms Lower'], inplace=True)


# ─────────────────────────────────────────────────────────────
# 3. GARAGE Y/N
# ─────────────────────────────────────────────────────────────
# Where Garage Y/N is missing, use Attached Garage value; else "No"
df['Garage Y/N'] = df['Garage Y/N'].fillna(df['Attached Garage']).fillna('No')
df['Garage Y/N'] = df['Garage Y/N'].map({'Yes': 1, 'No': 0})
df.drop(columns=['Attached Garage'], inplace=True)


# ─────────────────────────────────────────────────────────────
# 4. AIR CONDITIONING Y/N  (1 missing row → delete)
# ─────────────────────────────────────────────────────────────
df.dropna(subset=['Air Conditioning Y/N'], inplace=True)
df['Air Conditioning Y/N'] = df['Air Conditioning Y/N'].map({'Yes': 1, 'No': 0})


# ─────────────────────────────────────────────────────────────
# 5. BATHROOMS  (1 missing row → delete)
# ─────────────────────────────────────────────────────────────
df.dropna(subset=['Bathrooms'], inplace=True)
# Already floats — no further conversion needed


# ─────────────────────────────────────────────────────────────
# 6. TOTAL PARKING  (fill with median rounded down → 2)
# ─────────────────────────────────────────────────────────────
parking_median_floor = math.floor(df['Total Parking'].median())
df['Total Parking'] = df['Total Parking'].fillna(parking_median_floor).astype(int)


# ─────────────────────────────────────────────────────────────
# 7. SITE FEATURES  (missing → "None"; standardise item names)
# ─────────────────────────────────────────────────────────────
df['Site Features'] = df['Site Features'].fillna('None')

def normalise_site_features(value):
    if pd.isna(value) or value.strip().lower() == 'none':
        return 'None'
    items = sorted(set(item.strip() for item in value.split(',') if item.strip()))
    return ', '.join(items)

df['Site Features'] = df['Site Features'].apply(normalise_site_features)


# ─────────────────────────────────────────────────────────────
# 8. INTERIOR FEATURES  (missing → "None"; fix truncations, standardise names)
# ─────────────────────────────────────────────────────────────
df['Interior Features'] = df['Interior Features'].fillna('None')

# Truncated / variant entries to fix before keeping as categories
interior_fix_map = {
    # Truncations
    'fi': None, 'fir': None, 'fire': None, 'firepl': None,
    'v': None, 'vaulted c': None, 'vaulted ce': None, 'vaulted ceili': None, 'vaulted ceiling': None,
    'w': None, 'wa': None, 'wal': None, 'walk': None, 'walk-': None, 'walk-i': None, 'walk-in': None,
    'walk-in clos': None, 'walk-in close': None, 'walk-in closet(s': None,
    'walk-in pa': None, 'walk-in pant': None, 'walk-in pantr': None,
    'wall': None, 'wall to': None, 'wall to wa': None, 'wall to wall c': None,
    'wall to wall ca': None, 'wall to wall car': None, 'wall to wall carp': None,
    'wat': None, 'water h': None, 'water he': None, 'water hea': None, 'water heat': None, 'water heate': None,
    'wi': None, 'wir': None, 'wire': None, 'wired f': None, 'wired for gener': None,
    's': None, 'high t': None, 'skyligh': None, 'solarium/atr': None,
    'spr': None, 'sprinkl': None, 'sprinkler': None,
    'security sy': None, 'wine cell': None,
    # Variants
    'fir/softwood': 'Fir/Softwood',
    'walk-in closet(s)': 'Walk-In Closet(s)',
    'walk-in pantry': 'Walk-In Pantry',
    'vaulted ceiling(s)': 'Vaulted Ceiling(s)',
    'wall to wall carpet': 'Wall to Wall Carpet',
    'water heater': 'Water Heater',
    'wired for generator': 'Wired for Generator',
    'skylight(s)': 'Skylight(s)',
    'solarium/atrium': 'Solarium/Atrium',
    'sprinkler system': 'Sprinkler System',
    'security system': 'Security System',
    'wine cellar': 'Wine Cellar',
    'high tech cabling': 'High Tech Cabling',
    'fireplace (primary bedroom)': 'Fireplace (Primary Bedroom)',
}

def normalise_interior_features(value):
    if pd.isna(value) or value.strip().lower() == 'none':
        return 'None'
    clean_items = []
    for raw in value.split(','):
        item = raw.strip()
        lower = item.lower()
        mapped = interior_fix_map.get(lower, item)  # None means truncated junk → skip
        if mapped:
            clean_items.append(mapped)
    return ', '.join(sorted(set(clean_items))) if clean_items else 'None'

df['Interior Features'] = df['Interior Features'].apply(normalise_interior_features)


# ─────────────────────────────────────────────────────────────
# 9. SECURITY FEATURES  (missing → "None"; standardise names)
# ─────────────────────────────────────────────────────────────
df['Security Features'] = df['Security Features'].fillna('None')

security_name_map = {
    'fullyfenced': 'Fully Fenced',
    'partiallyfenced': 'Partially Fenced',
    'securitysystem': 'Security System',
    'securitygate': 'Security Gate',
    'securityservice': 'Security Service',
    'securityservices': 'Security Service',
    'fully fenced': 'Fully Fenced',
    'partially fenced': 'Partially Fenced',
    'security system': 'Security System',
    'security gate': 'Security Gate',
    'security service': 'Security Service',
}

def normalise_security_features(value):
    if pd.isna(value) or value.strip().lower() == 'none':
        return 'None'
    clean_items = []
    for raw in value.split(','):
        key = raw.strip().lower().replace(' ', '')
        # Try both spaced and non-spaced lookups
        canonical = security_name_map.get(raw.strip().lower()) or security_name_map.get(key)
        if canonical:
            clean_items.append(canonical)
    return ', '.join(sorted(set(clean_items))) if clean_items else 'None'

df['Security Features'] = df['Security Features'].apply(normalise_security_features)


# ─────────────────────────────────────────────────────────────
# 10. HOUSE SIZE
# ─────────────────────────────────────────────────────────────
# Remove commas and "SQFT", convert to int
df['house_size'] = (
    df['house_size']
    .str.replace(',', '', regex=False)
    .str.replace('SQFT', '', regex=False)
    .str.strip()
    .astype(int)
)


# ─────────────────────────────────────────────────────────────
# 11. TAX ANNUAL AMOUNT  (fill with mean)
# ─────────────────────────────────────────────────────────────
df['Tax Annual Amount'] = (
    df['Tax Annual Amount']
    .str.replace(r'[\$,\s]', '', regex=True)
    .replace('', np.nan)
    .astype(float)
)
tax_mean = df['Tax Annual Amount'].mean()
df['Tax Annual Amount'] = df['Tax Annual Amount'].fillna(tax_mean)


# ─────────────────────────────────────────────────────────────
# 12. YEAR BUILT → HOUSE AGE
# ─────────────────────────────────────────────────────────────
df['House Age'] = 2026 - df['Year Built']
df.drop(columns=['Year Built'], inplace=True)


# ─────────────────────────────────────────────────────────────
# 13. MLS AREA  (keep numeric code only)
# ─────────────────────────────────────────────────────────────
df['MLS Area'] = df['MLS Area'].str.extract(r'^(\d+)').astype(int)


# ─────────────────────────────────────────────────────────────
# 14. LEVELS  (pull from Stories Count where missing, else mode; standardise to float)
# ─────────────────────────────────────────────────────────────
levels_std_map = {
    'One': 1.0, 'Two': 2.0, 'Three Or More': 3.0,
    'One and One Half': 1.5, 'OneAndOneHalf': 1.5,
    'Multi/Split': 2.0,
}

# Stories Count has only the value 2.0 in this dataset
stories_to_levels = {2.0: 2.0}

def fill_levels(row):
    val = row['Levels']
    if pd.isna(val):
        sc = row['Stories Count']
        if not pd.isna(sc):
            return stories_to_levels.get(sc, np.nan)
        return np.nan
    return levels_std_map.get(val, np.nan)

df['Levels'] = df.apply(fill_levels, axis=1)

# Fill remaining NaN with mode (1.0)
levels_mode = df['Levels'].mode()[0]
df['Levels'] = df['Levels'].fillna(levels_mode)
df.drop(columns=['Stories Count'], inplace=True)


# ─────────────────────────────────────────────────────────────
# 15. LOT SIZE  (remove "Acres", convert to float)
# ─────────────────────────────────────────────────────────────
# All values are already in Acres — strip the unit and convert
df['lot_size'] = (
    df['lot_size']
    .str.replace('Acres', '', regex=False)
    .str.strip()
    .astype(float)
)


# ─────────────────────────────────────────────────────────────
# 16. FLOORING  (fill Unknown; standardise names; keep as category string)
# ─────────────────────────────────────────────────────────────
df['Flooring'] = df['Flooring'].fillna('Unknown')

flooring_name_map = {
    'Engineered Hardwood': 'Hardwood',
    'Laminate Hardwood': 'Laminate',
    'Laminate Tile': 'Laminate',
    'Bamboo/Cork': 'Bamboo',
    'See Remarks': 'Unknown',
    'Other': 'Unknown',
}

def normalise_flooring(value):
    if pd.isna(value) or value.strip().lower() == 'unknown':
        return 'Unknown'
    clean_items = []
    for raw in value.split(','):
        item = raw.strip()
        item = flooring_name_map.get(item, item)
        clean_items.append(item)
    return ', '.join(sorted(set(clean_items))) if clean_items else 'Unknown'

df['Flooring'] = df['Flooring'].apply(normalise_flooring)


# ─────────────────────────────────────────────────────────────
# 17. HOA  (fill missing with "No"; convert to binary 0/1)
# ─────────────────────────────────────────────────────────────
def hoa_to_binary(val):
    if pd.isna(val):
        return 0
    s = str(val).strip().lower()
    if s in ('no', '0', ''):
        return 0
    if s == 'yes':
        return 1
    try:
        return 0 if float(s) == 0 else 1
    except ValueError:
        return 0

df['HOA'] = df['HOA'].apply(hoa_to_binary)


# ─────────────────────────────────────────────────────────────
# 18. HOA FEE  (fill 0; remove $ and whitespace; convert to float)
# ─────────────────────────────────────────────────────────────
df['HOA Fee'] = (
    df['HOA Fee']
    .fillna('$0')
    .str.replace(r'[\$,\s]', '', regex=True)
    .replace('', '0')
    .astype(float)
)


# ─────────────────────────────────────────────────────────────
# 19. PROPERTY CONDITION  (fill Unknown; standardise; map to ordinal)
# ─────────────────────────────────────────────────────────────
df['Property Condition'] = df['Property Condition'].fillna('Unknown')

property_condition_map = {
    'Fixer': 1,
    'Fair': 2,
    'Average': 3,
    'Resale': 3,
    'Good': 4,
    'Restored': 4,
    'Very Good': 5,
    'Remodeled': 5,
    'Updated/Remodeled': 5,
    'Under Construction': 5,
    'Unknown': 0,
}

df['Property Condition'] = df['Property Condition'].map(property_condition_map).fillna(0).astype(int)


# ─────────────────────────────────────────────────────────────
# 20. FOUNDATION DETAILS  (fill Unknown; standardise names; keep as category string)
# ─────────────────────────────────────────────────────────────
df['Foundation Details'] = df['Foundation Details'].fillna('Unknown')

foundation_name_map = {
    'Concrete Slab': 'Slab',
    'See Remarks': 'Unknown',
}

def normalise_foundation(value):
    if pd.isna(value) or value.strip().lower() == 'unknown':
        return 'Unknown'
    clean_items = []
    for raw in value.split(','):
        item = raw.strip()
        item = foundation_name_map.get(item, item)
        clean_items.append(item)
    return ', '.join(sorted(set(clean_items))) if clean_items else 'Unknown'

df['Foundation Details'] = df['Foundation Details'].apply(normalise_foundation)


# ─────────────────────────────────────────────────────────────
# 21. LOT FEATURES  (fill None; standardise names; keep as category string)
# ─────────────────────────────────────────────────────────────
df['Lot Features'] = df['Lot Features'].fillna('None')

lot_feature_name_map = {
    'Drought Res Landscape': 'Drought Resistant Landscape',
}

def normalise_lot_features(value):
    if pd.isna(value) or value.strip().lower() == 'none':
        return 'None'
    clean_items = []
    for raw in value.split(','):
        item = raw.strip()
        item = lot_feature_name_map.get(item, item)
        clean_items.append(item)
    return ', '.join(sorted(set(clean_items))) if clean_items else 'None'

df['Lot Features'] = df['Lot Features'].apply(normalise_lot_features)


# ─────────────────────────────────────────────────────────────
# FINAL: Keep only cleaned target columns
# ─────────────────────────────────────────────────────────────
final_cols = [
    'Total Bedrooms', 'Garage Y/N', 'Air Conditioning Y/N', 'Bathrooms',
    'Total Parking', 'Inclusions', 'Site Features', 'Interior Features',
    'Security Features', 'house_size', 'Tax Annual Amount', 'House Age',
    'MLS Area', 'Levels', 'lot_size', 'Flooring', 'HOA', 'HOA Fee',
    'Property Condition', 'Foundation Details', 'Lot Features',
]



Done. Shape: (4508, 21)

Columns: ['Total Bedrooms', 'Garage Y/N', 'Air Conditioning Y/N', 'Bathrooms', 'Total Parking', 'Inclusions', 'Site Features', 'Interior Features', 'Security Features', 'house_size', 'Tax Annual Amount', 'House Age', 'MLS Area', 'Levels', 'lot_size', 'Flooring', 'HOA', 'HOA Fee', 'Property Condition', 'Foundation Details', 'Lot Features']

Sample (first 3 rows):
   Total Bedrooms  Garage Y/N  Air Conditioning Y/N  Bathrooms  Total Parking                                                                                        Inclusions                                                                                                           Site Features                                                                                                                   Interior Features Security Features  house_size  Tax Annual Amount  House Age  MLS Area  Levels  lot_size                 Flooring  HOA  HOA Fee  Property Condition Foundation Details                 